In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
BASE     = Path("..")
DISP_DIR = BASE / "dispersion"
OUT_DIR  = BASE / "skill_luck_decomposition"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LEAGUE_MAP = {
    "bundesliga": "Bundesliga",
    "la_liga": "La Liga",
    "premier_league": "Premier League",
    "serie_a": "Serie A",
    "Bundesliga": "Bundesliga",
    "La Liga": "La Liga",
    "Premier League": "Premier League",
    "Serie A": "Serie A",
}

def canon_league(name: str) -> str:
    return LEAGUE_MAP.get(str(name).strip(), str(name).strip())

In [3]:
# Helper to compute q for vectors (pandas Series)

# Using the formula: actual = q*skill + (1-q)*luck  =>  q = (actual - luck)/(skill - luck)

def compute_q_series(actual, skill, luck):
    denom = skill - luck
    q = (actual - luck) / denom

    # replace the value if denominator = 0 and values tending to infinity
    q = q.replace([np.inf, -np.inf], np.nan)
    
    return q.clip(0, 1)

In [6]:
path_actual = DISP_DIR / "actual" / "actual_dispersion.csv"
df_actual = pd.read_csv(path_actual)
print(df_actual.head())

df_actual["league"] = df_actual["league"].apply(canon_league)

df_actual = df_actual.rename(
    columns={
        "std_win_percentage": "std_win_pct_actual",
        "num_data_points": "num_data_points",
    }
)[["league", "std_win_pct_actual", "num_data_points"]]

           league  std_win_percentage  num_data_points
0      bundesliga            0.141848              378
1         la_liga            0.141384              460
2  premier_league            0.153524              440
3         serie_a            0.150573              454


In [8]:
path_skill = DISP_DIR / "pure_skill" / "skill_dispersion.csv"
df_skill = pd.read_csv(path_skill)
print(df_skill.head())


df_skill["league"] = df_skill["league"].apply(canon_league)

df_skill = df_skill.rename(
    columns={"std_win_percentage": "std_win_pct_skill"}
)[["league", "std_win_pct_skill"]]

           league  std_win_percentage  num_data_points
0      Bundesliga            0.305184              378
1         La Liga            0.303488              460
2  Premier League            0.303488              440
3         Serie A            0.303691              454


In [9]:
path_luck_res = DISP_DIR / "pure_luck_results_based" / "luck_results_based_dispersion.csv"
df_luck_res = pd.read_csv(path_luck_res)
print(df_luck_res.head())

df_luck_res["league"] = df_luck_res["league"].apply(canon_league)

df_luck_res = df_luck_res.rename(
    columns={"average_win_percentage": "std_win_pct_luck_results_avg"}
)[["league", "std_win_pct_luck_results_avg"]]

           league  win_percentage_seed_1  win_percentage_seed_2  \
0      Bundesliga               0.076200               0.073178   
1         La Liga               0.068648               0.068204   
2  Premier League               0.073299               0.068016   
3         Serie A               0.072642               0.067341   

   win_percentage_seed_3  win_percentage_seed_4  win_percentage_seed_5  \
0               0.071256               0.075025               0.069647   
1               0.070261               0.067590               0.070586   
2               0.070100               0.069416               0.071812   
3               0.067503               0.066312               0.067364   

   win_percentage_seed_6  win_percentage_seed_7  win_percentage_seed_8  \
0               0.069229               0.077884               0.076370   
1               0.076532               0.070419               0.069150   
2               0.071831               0.071942               0.070155 

In [10]:
path_luck_goals = DISP_DIR / "pure_luck_goals_based" / "luck_goals_based_dispersion.csv"
df_luck_goals = pd.read_csv(path_luck_goals)
print(df_luck_goals.head())

df_luck_goals["league"] = df_luck_goals["league"].apply(canon_league)

df_luck_goals = df_luck_goals.rename(
    columns={"average_win_percentage": "std_win_pct_luck_goals_avg"}
)[["league", "std_win_pct_luck_goals_avg"]]

           league  win_percentage_seed_1  win_percentage_seed_2  \
0      Bundesliga               0.072350               0.073379   
1         La Liga               0.072740               0.069078   
2  Premier League               0.070627               0.071237   
3         Serie A               0.067783               0.070712   

   win_percentage_seed_3  win_percentage_seed_4  win_percentage_seed_5  \
0               0.074869               0.074032               0.071813   
1               0.071992               0.071072               0.069485   
2               0.068880               0.071134               0.073381   
3               0.070864               0.072115               0.071405   

   win_percentage_seed_6  win_percentage_seed_7  win_percentage_seed_8  \
0               0.072987               0.076851               0.075271   
1               0.071851               0.068953               0.067849   
2               0.067073               0.073971               0.072666 

In [11]:
wide = (
    df_actual
    .merge(df_skill,      on="league", how="inner")
    .merge(df_luck_res,   on="league", how="inner")
    .merge(df_luck_goals, on="league", how="inner")
)

wide["q_results"] = compute_q_series(
    wide["std_win_pct_actual"],
    wide["std_win_pct_skill"],
    wide["std_win_pct_luck_results_avg"],
)

wide["q_goals"] = compute_q_series(
    wide["std_win_pct_actual"],
    wide["std_win_pct_skill"],
    wide["std_win_pct_luck_goals_avg"],
)

wide = wide[
    [
        "league",
        "num_data_points",
        "std_win_pct_actual",
        "std_win_pct_skill",
        "std_win_pct_luck_results_avg",
        "std_win_pct_luck_goals_avg",
        "q_results",
        "q_goals",
    ]
].sort_values("league")

wide

,league,num_data_points,std_win_pct_actual,std_win_pct_skill,std_win_pct_luck_results_avg,std_win_pct_luck_goals_avg,q_results,q_goals
0,Bundesliga,378,0.141848,0.305184,0.073866,0.074277,0.293893,0.292635
1,La Liga,460,0.141384,0.303488,0.070224,0.069876,0.305063,0.306097
2,Premier League,440,0.153524,0.303488,0.071076,0.070879,0.354750,0.355294
3,Serie A,454,0.150573,0.303691,0.068512,0.070149,0.348931,0.344366


In [12]:
out_path = OUT_DIR / "dispersion_skill_luck_decomposition.csv"
wide.to_csv(out_path, index=False)
print(f"Saved dispersion skill–luck decomposition to: {out_path}")

Saved dispersion skill–luck decomposition to: ../skill_luck_decomposition/dispersion_skill_luck_decomposition.csv
